In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

## Funções de auditoria

Funções reutilizadas ao longo do notebook para relatório inicial (pré-avaliação) e relatório final (comparativo antes/depois) de cada base.

In [ ]:
def relatorio_inicial(df):
    """
    PRE AVALIACAO DA BASE
    """
    print("=" * 48)
    print(f"ETAPA 1 - AUDITORIA")
    print("=" * 48)
    
    print("\n[1] COLUNAS DA BASE:")
    print(df.columns.tolist())
    
    print("\n[2] INFO DA BASE:")
    df.info()
    
    print("\n[3] TIPOS DE DADOS (dtypes):")
    print(df.dtypes)
    
    print("\n[4] ESTATÍSTICAS DESCRITIVAS (Numéricas):")
    print(df.describe())

    '''print("\n[5] ESTATÍSTICAS DESCRITIVAS (Objetos/Texto): ")
    try:
        print(df.describe(include='object'))
    except ValueError:
        print("Não há colunas de texto nesta base.")'''
    
    print("\n[6] QUANTIDADE DE NULOS POR COLUNA:")
    print(df.isnull().sum())
     
    print("\n[8] PERCENTUAL DE NULOS (> 15%):")
    percentual_nulos = (df.isnull().sum() / len(df) * 100).round(1)
    print(percentual_nulos)
        
    print("\n[9] LINHAS COM VALORES NULOS:")
    qtd_nulos_linhas = df.isnull().any(axis=1).sum()
    print(f"Total de linhas com pelo menos um valor nulo: {qtd_nulos_linhas}")
    
    print("\n[10] TOTAL DE LINHAS DUPLICADAS:")
    total_duplicadas = df.duplicated().sum()
    print(f"Total: {total_duplicadas}")
    
    print("\n" + "=" * 48)
    print(f"FIM DO RELATÓRIO")
    print("=" * 48 + "\n")

def relatorio_final(df_original, df_atual):
    """
    RELATÓRIO FINAL DE AUDITORIA
    """
    registros_iniciais = df_original.shape[0]
    registros_finais = df_atual.shape[0]
    colunas_iniciais = df_original.shape[1]
    colunas_finais = df_atual.shape[1]

    print('=' * 48)
    print('      RELATÓRIO FINAL DE AUDITORIA')
    print('=' * 48)
    print(f'Registros iniciais:     {registros_iniciais:>6}')
    print(f'Registros finais:       {registros_finais:>6}')
    print(f'Registros removidos:    {registros_iniciais - registros_finais:>6}')
    print(f'Colunas originais:      {colunas_iniciais:>6}')
    print(f'Colunas finais:         {colunas_finais:>6}')
    print('=' * 48)
    print(f'Nulos restantes:        {df_atual.isnull().sum().sum():>6}')
    print(f'Duplicatas restantes:   {df_atual.duplicated().sum():>6}')
    print('=' * 48)

## Convertendo as bases em DataFrame

Leitura dos CSVs originais e concatenação com as bases complementares (clientes, quartos, funcionários, reservas e unidades novos).

In [ ]:
canais = pd.read_csv('canais_venda.csv', sep=';', encoding='utf-8')
clientes = pd.read_csv('clientes.csv', sep=';', encoding='utf-8')
funcionarios = pd.read_csv('funcionarios.csv', sep=';', encoding='utf-8')
reservas = pd.read_csv('reservas.csv', sep=';', encoding='utf-8')
quartos = pd.read_csv('tipos_quarto.csv', sep=';', encoding='utf-8')
unidades = pd.read_csv('unidades.csv', sep=';', encoding='utf-8')


clientes_novos=pd.read_csv('clientes_novos.csv', sep=';', encoding='utf-8')
df_concatenado_clientes = pd.concat([clientes, clientes_novos], ignore_index=True)

quartos_novos=pd.read_csv('tipos_quarto_novos.csv', sep=';', encoding='utf-8')
df_concatenado_quartos = pd.concat([quartos, quartos_novos], ignore_index=True)

funcionarios_novos=pd.read_csv('funcionarios_novos.csv', sep=';', encoding='utf-8')
df_concatenado_funcionarios = pd.concat([funcionarios, funcionarios_novos], ignore_index=True)

reservas_novos=pd.read_csv('reservas_novas.csv', sep=';', encoding='utf-8')
df_concatenado_reservas = pd.concat([reservas, reservas_novos], ignore_index=True)

unidades_total=pd.read_csv('unidades_novos.csv', sep=';', encoding='utf-8')

df_concatenado_unidades = pd.concat([unidades, unidades_total], ignore_index=True)

In [ ]:
canais_original = canais.copy()
df_concatenado_unidades_original = df_concatenado_unidades.copy()
df_concatenado_clientes_original = df_concatenado_clientes.copy()
df_concatenado_funcionarios_original = df_concatenado_funcionarios.copy()
df_concatenado_quartos_original = df_concatenado_quartos.copy()
df_concatenado_reservas_original = df_concatenado_reservas.copy()

* Criando as cópias para o relatório final/arquivo

## Etapa 1 — Auditoria

Aplica a função `relatorio_inicial` para identificar inconsistências antes da limpeza em todas as bases.

In [ ]:
relatorio_inicial (canais)


In [ ]:
relatorio_inicial (funcionarios)


In [ ]:
relatorio_inicial(reservas)


In [ ]:
relatorio_inicial(quartos)

In [ ]:
relatorio_inicial(unidades)


In [ ]:
relatorio_inicial(clientes_novos)

## 2. Limpeza e tratamento

### 2.1 Base: funcionários

Correção da base salarial via `map`.

In [ ]:
novos_salarios = {
    9: '1870.00',
    10:'2314.00',
    14: '5480.00',
    37: '2950.00',
    43:'5600.00',
    92:'8600.00',
    104:'2890.00',
    11:'2800.00',
    26:'2150.00',
}

df_concatenado_funcionarios.loc[df_concatenado_funcionarios['id_funcionario'].isin(novos_salarios), 'salario'] = df_concatenado_funcionarios['id_funcionario'].map(novos_salarios)

In [ ]:
df_concatenado_funcionarios['departamento'].value_counts()

* Padronização das categorias de `departamento` via `map`.

In [ ]:
df_concatenado_funcionarios['departamento'].value_counts()

mapa = {
    'Gov.':                     'Governança',
    'Admin.':  'Administração'             ,
    'operações' :'Operações',
    'A&B': 'Alimentos e Bebidas'

}

df_concatenado_funcionarios['departamento'] = df_concatenado_funcionarios['departamento'].map(mapa).fillna(df_concatenado_funcionarios['departamento'])#joga as informações do dicionario mapa no df. filne mantem os valores corretos que não precisam de substituição

print('Após mapeamento:\n')
print(sorted(df_concatenado_funcionarios['departamento'].unique()))

Padronização de strings na coluna `salario`.

In [ ]:
df_concatenado_funcionarios['salario'] = (
    df_concatenado_funcionarios['salario']
    .str.replace('"', '', regex=False)
    .str.replace("'","", regex=False)
    .str.replace(",",".", regex=False))

* Conversão de `data_admissao` para `datetime`
* Aplicação da correção de `salario`
* Padronização de `cargo`

In [ ]:
df_concatenado_funcionarios['data_admissao'] = pd.to_datetime(
    df_concatenado_funcionarios['data_admissao'],
    format='mixed',
    errors='coerce'
)

def corrigir_valor(x):
    if pd.isna(x):
        return x
    x = str(x)
    if x.count('.') == 2:
        x = x.replace('.', '', 1)
    return x

df_concatenado_funcionarios['salario'] = df_concatenado_funcionarios['salario'].apply(corrigir_valor)

df_concatenado_funcionarios['salario'] = pd.to_numeric(df_concatenado_funcionarios['salario'], errors='coerce')

df_concatenado_funcionarios[df_concatenado_funcionarios['salario'].isna()][['nome', 'salario']]

novos_salarios = {
    142: 7850.00    
}

df_concatenado_funcionarios.loc[
    df_concatenado_funcionarios['id_funcionario']
    .isin(novos_salarios),'salario'
] = df_concatenado_funcionarios['id_funcionario'].map(novos_salarios)

df_concatenado_funcionarios['cargo'] = df_concatenado_funcionarios['cargo'].str.upper()

df_concatenado_funcionarios.head(50)  

* Relatório Final

In [ ]:
relatorio_final(df_concatenado_unidades_original, df_concatenado_funcionarios)

### 2.2 Base: reservas

* Correção de diárias e avaliações
* Correção de hóspedes

Por decisão do cliente, valores nulos/impossíveis de `avaliacao_hospede` são mantidos como não avaliados. Os valores não informados somam 646 registros (13,9% do total de avaliações).

In [ ]:
relatorio_inicial(df_concatenado_reservas)

* Inclusão e correção de valores fornecidos pelo cliente via `map`, e verificação de valores negativos em `num_hospedes` e `qtd_diarias`.

In [ ]:
novos_valores = {
    728: 7, 955: 4, 1134: 3, 1152: 2,
    1448: 6, 1581: 4, 2312: 4, 2384: 2,
    2509: 1, 4030: 1, 3472: 1, 3505: 1, 
    3838: 1, 4048: 1
}
novas_avaliacoes = {
    37: 1, 902: 1, 2073: 1, 933: 3, 1718: 3, 2167: 3,
    3541: 1, 2874: 1, 4127: 1, 4088: 3
}

correcao_novos_hospedes = {
    568: 1, 675: 2, 903: 1, 1201: 2, 1883: 2,
    2427: 2, 4463: 1, 4506: 2, 4040: 2, 2687: 2,
}

ids_hospedes = list(correcao_novos_hospedes.keys())
df_concatenado_reservas.loc[
    df_concatenado_reservas['id_reserva'].isin(ids_hospedes), 'num_hospedes'
] = df_concatenado_reservas['id_reserva'].map(correcao_novos_hospedes)

ids_avaliacoes = list(novas_avaliacoes.keys())

df_concatenado_reservas.loc[
    df_concatenado_reservas['id_reserva']
    .isin(ids_avaliacoes),
    'avaliacao_hospede'
] = df_concatenado_reservas['id_reserva'].map(novas_avaliacoes)

ids_valores = list(novos_valores.keys())

df_concatenado_reservas.loc[
    df_concatenado_reservas['id_reserva']
    .isin(ids_valores),
    'qtd_diarias'] = df_concatenado_reservas['id_reserva'].map(novos_valores)

df_filtrado = df_concatenado_reservas[df_concatenado_reservas['id_reserva'].isin(ids_valores)]

print((df_concatenado_reservas['num_hospedes'] < 0).sum())
print((df_concatenado_reservas['qtd_diarias'] < 0).sum())
print((df_concatenado_reservas['id_canal'] == -1).sum())

In [ ]:
df_concatenado_reservas[df_concatenado_reservas['avaliacao_hospede'].isnull()]

* Identificação de duplicidades (inspeção com `value_counts()`).

In [ ]:

print(df_concatenado_reservas['id_reserva'].value_counts())


* Remoção de duplicidades em `id_reserva`, mantendo a primeira ocorrência (`drop_duplicates`, `keep='first'`).

In [ ]:
shape_inicial = df_concatenado_reservas.shape[0]

df_concatenado_reservas = df_concatenado_reservas.drop_duplicates(
    subset=['id_reserva'], 
    keep='first'
)

shape_final = df_concatenado_reservas.shape[0]

print(f'Registros iniciais: {shape_inicial}')
print(f'Registros finais: {shape_final}')
print(f'{shape_inicial - shape_final} registros duplicados apagados')

* Conversão de `id_canal` para `int`
* Conversão de `data_checkin` e `data_checkout` de `str` para `datetime`

In [ ]:

df_concatenado_reservas = df_concatenado_reservas.dropna(subset=['id_canal'])

df_concatenado_reservas['id_canal'] = df_concatenado_reservas['id_canal'].astype(int)

df_concatenado_reservas['avaliacao_hospede'] = (
    pd.to_numeric(df_concatenado_reservas['avaliacao_hospede'], errors='coerce')
)

* Padronização de strings nas colunas `status_reserva` e `forma_pagamento`.

In [ ]:
df_concatenado_reservas['status_reserva'] = (
    df_concatenado_reservas['status_reserva']
    .str.strip()
    .str.upper()
)
df_concatenado_reservas['forma_pagamento'] = (
    df_concatenado_reservas['forma_pagamento']
    .str.strip()
    .str.upper()
)
df_concatenado_reservas[['forma_pagamento', 'status_reserva']].head(10)

* Padronização de `forma_pagamento` via `map`.

In [ ]:
mapa_status = {
    'CONFIRMADA': 'CONFIRMADA',
    'CONF.': 'CONFIRMADA',
    'CANCELADA': 'CANCELADA',
    'CANCEL.': 'CANCELADA',
    'CONCLUÃDA': 'CONCLUÍDA',
    'CONCLUIDA': 'CONCLUÍDA',
    'NO-SHOW': 'NO-SHOW'
}

df_concatenado_reservas['status_reserva'] = (
    df_concatenado_reservas['status_reserva']
    .map(mapa_status)
)
print(df_concatenado_reservas['status_reserva'].value_counts())

* Tratamento de nulos com `fillna()`. Decisão de não excluir registros (cadastro de clientes e reservas), mantendo apenas `id_canal`s válidos para a integração com o SQL.

In [ ]:
mapa_pagamento = {
    'DINHEIRO': 'DINHEIRO',
    'CASH': 'DINHEIRO',

    'PIX': 'PIX',

    'CARTÃO DE DÉBITO': 'CARTÃO DE DÉBITO',
    'CARTAO DEBITO': 'CARTÃO DE DÉBITO',
    'CD': 'CARTÃO DE DÉBITO',
    'DÉBITO': 'CARTÃO DE DÉBITO',

    'CARTÃO DE CRÉDITO': 'CARTÃO DE CRÉDITO',
    'CARTAO CREDITO': 'CARTÃO DE CRÉDITO',
    'C. CRÉDITO': 'CARTÃO DE CRÉDITO',
    'CRED': 'CARTÃO DE CRÉDITO',
    'CC': 'CARTÃO DE CRÉDITO',

    'TRANSFERÊNCIA': 'TRANSFERÊNCIA',
    'TRANSFERENCIA': 'TRANSFERÊNCIA',
    'TED': 'TRANSFERÊNCIA'
}

df_concatenado_reservas['forma_pagamento'] = (
    df_concatenado_reservas['forma_pagamento']
    .map(mapa_pagamento)
)
print(df_concatenado_reservas['forma_pagamento'].value_counts())

* Tratando nulos: recurso *fillna()* para o preenchimento de valores não informados

* Decisão de não exclusão devido ao tipo de dado (cadastro de clientes e reservas)

* Tratamento dos id_canais apenas válidos para resolver a integração entre base e SQL.

In [ ]:
df_concatenado_reservas['id_canal'] = (
    df_concatenado_reservas['id_canal']
    .fillna(-1)
)

df_concatenado_reservas['forma_pagamento'] = (
    df_concatenado_reservas['forma_pagamento']
    .fillna('NÃO INFORMADO')
)

df_concatenado_reservas['status_reserva'] = (
    df_concatenado_reservas['status_reserva']
    .fillna('NÃO INFORMADO')
)

ids_validos_canais = canais['id_canal'].unique()
df_concatenado_reservas = df_concatenado_reservas[df_concatenado_reservas['id_canal'].isin(ids_validos_canais)]

ids_validos_canais
print(df_concatenado_reservas.isnull().sum())

Correção da coluna `avaliacao_hospede`: conversão para numérico com `pd.to_numeric(errors='coerce')`, preservando `NaN` para não distorcer o cálculo de médias.

In [ ]:
df_concatenado_reservas['avaliacao_hospede'] = pd.to_numeric(
    df_concatenado_reservas['avaliacao_hospede'], 
    errors='coerce')

df_concatenado_reservas['avaliacao_hospede'].info()

print(df_concatenado_reservas['avaliacao_hospede'].isnull().sum())

* Correção da coluna `avaliacao_hospede`: conversão para numérico com `pd.to_numeric(errors='coerce')`, preservando `NaN` para não distorcer o cálculo de médias.

In [ ]:
df_concatenado_reservas['id_canal'] = df_concatenado_reservas['id_canal'].astype(int)

In [ ]:
df_concatenado_reservas['data_checkin'] = pd.to_datetime(
    df_concatenado_reservas['data_checkin'],
    format='mixed',
    errors='coerce'
)
df_concatenado_reservas['data_checkout'] = pd.to_datetime(
    df_concatenado_reservas['data_checkout'],
    format='mixed',
    errors='coerce'
)

print(df_concatenado_reservas[['data_checkin', 'data_checkout']].head(10))
print("\n")
print(df_concatenado_reservas[['data_checkin', 'data_checkout']].isnull().sum())

* Tratamento dos valores negativos em `qtd_diarias` e `num_hospedes`

In [ ]:
negativos_diarias = df_concatenado_reservas[df_concatenado_reservas['qtd_diarias'] < 0]
negativos_hospedes = df_concatenado_reservas[df_concatenado_reservas['num_hospedes'] < 0]

print(negativos_diarias)
print(negativos_hospedes)

Filtros finais de consistência:

* `num_hospedes`: mantém apenas reservas entre 1 e 20 hóspedes
* `qtd_diarias`: mantém apenas estadias entre 1 e 60 dias

Nulos intencionais em `avaliacao_hospede` são preservados.

In [ ]:
antes = df_concatenado_reservas.shape[0]

df_concatenado_reservas = df_concatenado_reservas[
    df_concatenado_reservas['num_hospedes'].between(1, 20) &
    df_concatenado_reservas['qtd_diarias'].between(1, 60)
]

total_nulos = df_concatenado_reservas['avaliacao_hospede'].isnull().sum()

print(f'Valores removidos (fora do intervalo): {antes - df_concatenado_reservas.shape[0]}')
print(f'Shape atual: {df_concatenado_reservas.shape}')
print(f"Total de avaliações nulas: {total_nulos}")

* Relatório Final

In [ ]:
relatorio_final(df_concatenado_reservas_original, df_concatenado_reservas)

### 2.3 Base: clientes

- `faixa_etaria` em formato `str`, com valores nulos
- Erros de grafia em `estado_origem`, `forma_pagamento` e `tipo_cliente`
- Duplicidade no `id_cliente` nº 47

Por decisão do cliente, os 29 registros vazios em `faixa_etaria` são mantidos como nulos, não como zero.

In [ ]:
relatorio_inicial(df_concatenado_clientes_original)

* Refinamento da auditoria com `value_counts()`: inconsistências de grafia e divergência entre `cidade_origem` e `estado_origem`.

In [ ]:
print("\n--- Auditoria de IDs negativos ---")
display(df_concatenado_clientes[df_concatenado_clientes['id_cliente'] < 0])

print("\n--- Auditoria da coluna Estado de Origem ---")
display(df_concatenado_clientes['estado_origem'].value_counts())

print("\n--- Auditoria da coluna Faixa Etária ---")
display(df_concatenado_clientes['faixa_etaria'].value_counts())

print("\n--- Auditoria da coluna Tipo de Cliente ---")
display(df_concatenado_clientes['tipo_cliente'].value_counts())

print("\n--- Auditoria da coluna Cidade de Origem ---")
display(df_concatenado_clientes['cidade_origem'].value_counts())

print("\n--- Auditoria de duplicidade de Id Cliente ---")
display(df_concatenado_clientes['id_cliente'].value_counts())

* Padronização da coluna `nome`.

In [ ]:
df_concatenado_clientes['nome'] = df_concatenado_clientes['nome'].str.strip().str.title()

df_concatenado_clientes['nome']

* Padronização da grafia de `cidade_origem` e correspondência com `estado_origem` a partir de um dicionário cidade→estado.

In [ ]:
df_concatenado_clientes['cidade_origem'] = (
    df_concatenado_clientes['cidade_origem']
    .str.strip()
    .str.upper()
)

mapa_cidade_estado = {
    'FORTALEZA': 'CE',
    'SÃO PAULO': 'SP',
    'RECIFE': 'PE',
    'CURITIBA': 'PR',
    'BRASÍLIA': 'DF',
    'BELO HORIZONTE': 'MG',
    'SALVADOR': 'BA',
    'RIO DE JANEIRO': 'RJ',
    'NITERÓI': 'RJ',
    'PORTO ALEGRE': 'RS',
    'LISBOA': 'EX',
    'BUENOS AIRES': 'EX',
    'NOVA YORK': 'EX',
    'MIAMI': 'EX',
}

df_concatenado_clientes['estado_origem'] = df_concatenado_clientes['cidade_origem'].map(mapa_cidade_estado)

print(df_concatenado_clientes[['cidade_origem', 'estado_origem']].isnull().sum())

df_concatenado_clientes['cidade_origem'].value_counts()


Padronização da coluna `tipo_cliente`.

In [ ]:
df_concatenado_clientes['tipo_cliente'] = df_concatenado_clientes['tipo_cliente'].str.strip().str.upper()

mapa_tipo_cliente = {
    'CORPORATIVO': 'CORPORATIVO',
    'PF': 'PESSOA FÍSICA',
    'PJ': 'PESSOA JURÍDICA',
    'PESSOA FISICA': 'PESSOA FÍSICA',
    'PESSOA FÍSICA': 'PESSOA FÍSICA',
    'PESSOA JURÍDICA': 'PESSOA JURÍDICA',
}

df_concatenado_clientes['tipo_cliente'] = df_concatenado_clientes['tipo_cliente'].map(mapa_tipo_cliente)

print(df_concatenado_clientes['tipo_cliente'].value_counts())

* Padronização final de `cidade_origem` (maiúscula inicial, sem espaços nas bordas).

In [ ]:
df_concatenado_clientes['cidade_origem'] = (
    df_concatenado_clientes['cidade_origem']
    .str.strip()
    .str.title()
)

df_concatenado_clientes['cidade_origem']

* Resolução da duplicidade do `id_cliente` nº 47: por recomendação do cliente, o `id_cliente` 401 é atribuído ao registro de Carlos Neto.

In [ ]:
df_concatenado_clientes['id_cliente'].value_counts()

df_concatenado_clientes.loc[
    (df_concatenado_clientes['id_cliente'] == 47)
    & (df_concatenado_clientes['nome'] == 'Carlos Neto'),
    'id_cliente'] = 401

print(df_concatenado_clientes['id_cliente'].value_counts())
print(df_concatenado_clientes[df_concatenado_clientes['id_cliente'].isin([47, 401])])

* Relatório Final

In [ ]:
relatorio_final(df_concatenado_clientes_original, df_concatenado_clientes)

### 2.4 Base: canais de venda

- Padronização da grafia de `nome_canal`
- Remoção do símbolo de percentual e conversão de `comissao_pct` para numérico

In [ ]:
relatorio_inicial(canais_original)

canais['nome_canal'] = canais['nome_canal'].str.strip().str.upper()

canais['comissao_pct'] = (
    canais['comissao_pct']
    .astype(str)
    .str.replace('%', '', regex=False)
)

canais['comissao_pct'] = pd.to_numeric(canais['comissao_pct'], errors='coerce')

* Relatório Final

In [ ]:
relatorio_final(canais_original, canais)

### 2.5 Base: unidades

Padronização das grafias de `regiao` e `categoria_hotel`.

In [ ]:
relatorio_inicial(df_concatenado_unidades_original)

df_concatenado_unidades['regiao'] = (
    df_concatenado_unidades['regiao']
    .replace('CAP', 'Capital')
    .str.strip()                
    .str.title()                
)

df_concatenado_unidades['categoria_hotel'] = (
    df_concatenado_unidades['categoria_hotel']
    .str.lower()                           
    .str.replace('cinco', '5')             
    .str.replace('três', '3')           
    .str.replace('*', ' estrelas')
    .str.title()
)

df_concatenado_unidades.tail(15)

* Relatório Final

In [ ]:
relatorio_final(df_concatenado_unidades_original, df_concatenado_unidades)

### 2.6 Base: tipos de quarto

Remoção de duplicatas e padronização da grafia de `descricao`.

In [ ]:
relatorio_inicial(df_concatenado_quartos_original)

In [ ]:
df_concatenado_quartos = df_concatenado_quartos.drop_duplicates(subset=['id_tipo_quarto'])
df_concatenado_quartos['descricao'] = df_concatenado_quartos['descricao'].str.strip().str.upper()

df_concatenado_quartos['descricao']

* Remoção de `R$`, aspas e espaços, e conversão de `valor_diaria_base` para numérico.

In [ ]:
df_concatenado_quartos['valor_diaria_base'] = (
    df_concatenado_quartos['valor_diaria_base']
    .astype(str)
    .str.replace('R$', '', regex=False)
    .str.replace('"', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
)
df_concatenado_quartos['valor_diaria_base'] = (
    pd.to_numeric(df_concatenado_quartos['valor_diaria_base'], 
                  errors='coerce'))

df_concatenado_quartos

* Preenchimento do total de quartos faltantes em `num_quartos_total`.

In [ ]:
quartos_faltantes = {
    7: 70.0,
    10: 75.0
}

df_concatenado_unidades['num_quartos_total'] = (
    df_concatenado_unidades['num_quartos_total']
    .fillna(df_concatenado_unidades['id_unidade'].map(quartos_faltantes))
)

df_concatenado_unidades['num_quartos_total'] = df_concatenado_unidades['num_quartos_total'].astype(int)

print(df_concatenado_unidades['num_quartos_total'].isnull().sum())

df_concatenado_unidades

* Relatório Final

In [ ]:
relatorio_final(df_concatenado_quartos_original, df_concatenado_quartos)

### 3. Exportando os CSVs tratados

In [ ]:
canais.to_csv('canais_tratada.csv', index=False)
df_concatenado_clientes.to_csv('clientes_tratada.csv', index=False, na_rep=r'\N')
df_concatenado_quartos.to_csv('quartos_tratada.csv', index=False)
df_concatenado_funcionarios.to_csv('funcionarios_tratada.csv', index=False)
df_concatenado_reservas.to_csv('reservas_tratada.csv', index=False, na_rep=r'\N')
df_concatenado_unidades.to_csv('unidades_tratada.csv', index=False)

## 4. Respondendo às perguntas de negócio

#### 1. Comportamento das diárias e distribuição do faturamento

Cruzamento de bases e cálculo de faturamento:

- Cruzamento de `reservas`, `quartos` e `unidades`. (Recursos: `pandas.merge`)
- Exclusão de diárias zeradas ou negativas. (Recursos: filtros booleanos)
- Cálculo de `faturamento` por reserva. (Recursos: operação vetorial)

Dicionário de Funções

In [ ]:
def converte_moeda(valor):
    return f"R$ {valor:,.2f}".replace(",", ".")

def acha_unidades_discrepantes(df):
    array_col_valores = np.array(df['faturamento'])

    q1 = np.percentile(array_col_valores, 25)
    q3 = np.percentile(array_col_valores, 75)
    iqr = q3 - q1

    limite_superior = q3 + (1.5 * iqr)
    limite_inferior = q1 - (1.5 * iqr)

    outlier_sup = df.loc[df['faturamento'] > limite_superior]
    outlier_inf = df.loc[df['faturamento'] < limite_inferior]
    
    return outlier_sup, outlier_inf

def classificar_correlacao(corr):
    """
    Classificação da Correlação de Pearson.
    """
    if 0.9 <= corr <= 1.0:
        return "Muito forte positiva"
    elif 0.7 <= corr < 0.9:
        return "Forte positiva"
    elif 0.4 <= corr < 0.7:
        return "Moderada positiva"
    elif -0.4 < corr < 0.4:
        return "Fraca / sem relação"
    elif -0.7 < corr <= -0.4:
        return "Moderada negativa"
    elif -0.9 < corr <= -0.7:
        return "Forte negativa"
    elif -1.0 <= corr <= -0.9:
        return "Muito forte negativa"
    else:
        return "Fora do limite de Pearson"

In [ ]:
df_reservas_total = (
    df_concatenado_reservas
    .merge(df_concatenado_quartos, on='id_tipo_quarto', how='left')
    .merge(df_concatenado_unidades, on='id_unidade', how='left')
)

df_reservas_total = df_reservas_total[df_reservas_total['valor_diaria_base'] > 0]

df_reservas_total['faturamento'] = (
    df_reservas_total['valor_diaria_base'] 
    * df_reservas_total['qtd_diarias']
)

df_reservas_total.columns

Aplicando filtros para mapeamento do negócio em escala regional
1) filtro região
2) filtro cidade
3) filtro por unidade

In [ ]:
faturamento_regiao = (
    df_reservas_total.groupby('regiao')['faturamento']
    .sum()
    .sort_values(ascending=False)
)

faturamento_cidade = (
    df_reservas_total.groupby('cidade')['faturamento']
    .sum()
    .sort_values(ascending=False)
)

faturamento_unidade = (
    df_reservas_total.groupby('nome_unidade')['faturamento']
    .sum()
    .sort_values(ascending=False)
)

print(" Faturamento Total por Região ")
display(faturamento_regiao.apply(converte_moeda))

print("\n Faturamento Total por Cidade ")
display(faturamento_cidade.apply(converte_moeda))

print("\n Faturamento Total por Unidade ")
display(faturamento_unidade.apply(converte_moeda))


* Aplicando filtro temporal: quais são os melhores meses por unidade ?

In [ ]:
df_reservas_total['mes'] = df_reservas_total['data_checkin'].dt.month

reservas_por_mes = (
    df_reservas_total.groupby(['nome_unidade', 'mes'])
    .size()
    .reset_index(name='qtd_reservas')
)

melhor_mes_por_unidade = (
    reservas_por_mes
    .sort_values(by=['nome_unidade', 'qtd_reservas'], ascending=[True, False])
    .drop_duplicates(subset=['nome_unidade'], keep='first')
)

print(" Mês com Mais Reservas por Unidade : ")
melhor_mes_por_unidade

#### 2. Capacidade operacional e impactos ao negócio

- Filtragem de reservas efetivamente realizadas (`CONFIRMADA`/`CONCLUÍDA`). (Recursos: `.isin()`)
- Comparação entre hóspedes e capacidade máxima do quarto. (Recursos: filtros lógicos)
- Cálculo do total de hóspedes excedentes por unidade.
- Ranqueamento das unidades com mais reservas acima da capacidade. (Recursos: `.groupby()`, `.size()`, `.sort_values()`)

In [ ]:
reservas_realizadas = df_reservas_total[
    df_reservas_total['status_reserva'].isin(['CONFIRMADA', 'CONCLUÍDA'])
]

acima_capacidade = reservas_realizadas[
    reservas_realizadas['num_hospedes'] > reservas_realizadas['capacidade_max']
]

hospedes_excedentes = acima_capacidade['num_hospedes'] - acima_capacidade['capacidade_max']

qtd_reservas_por_unidade = acima_capacidade.groupby('nome_unidade').size()

total_excedente_por_unidade = hospedes_excedentes.groupby(acima_capacidade['nome_unidade']).sum()

print("\nTotal de hóspedes excedentes, por unidade:")
print(total_excedente_por_unidade.sort_values(ascending=False))

print("\nQuantidade de reservas acima da capacidade, por unidade:")
print(qtd_reservas_por_unidade.sort_values(ascending=False))

#### 3. Performances discrepantes

Diagnóstico de faturamento e anomalias:

- Regras de decisão cruzando assimetria e curtose para categorizar o comportamento financeiro da rede. (Recursos: `if`/`elif`)
- Curtose para medir a severidade das anomalias apontadas pela assimetria, isolando extremos severos (caudas pesadas). (Recursos: `.kurt()`)

In [ ]:
df_faturamento = faturamento_unidade.reset_index()

df_faturamento.columns = ['nome_unidade', 'faturamento']

assimetria = df_faturamento['faturamento'].skew()

curtose = df_faturamento['faturamento'].kurt()

if assimetria < -0.5 and curtose > 0.5:
    print("Diagnóstico: Assimetria negativa com caudas pesadas.")
    print("Conclusão: Gargalos impactando o faturamento para baixo.\n")

elif assimetria > 0.5 and curtose > 0.5:
    print("Diagnóstico: Assimetria positiva com caudas pesadas.")
    print("Conclusão: Outliers superiores elevando o faturamento.\n")

elif assimetria < -0.5 and curtose <= 0.5:
    print("Diagnóstico: Assimetria negativa leve.")
    print("Conclusão: Faturamento geral com viés de baixa.\n")

elif assimetria > 0.5 and curtose <= 0.5:
    print("Diagnóstico: Assimetria positiva leve.")
    print("Conclusão: Faturamento geral com viés de alta.\n")

else:
    print("Diagnóstico: Distribuição simétrica.")
    print("Conclusão: Faturamento homogêneo na rede.\n")

print("-----Índices Estatísticos ----")
print(f"Skew (Assimetria) : {assimetria.round(2)}")
print(f"Kurt (Curtose)    : {curtose.round(2)}")

Isolamento de outliers pelo método IQR:

* Aplicação do limite interquartil para identificar as unidades com faturamento fora do padrão esperado. (Recursos: estatística descritiva / IQR)

* Lógica encapsulada em função própria, reaproveitada de outro projeto do curso. (Recursos: `def`)

In [ ]:
hoteis_acima_da_curva, hoteis_abaixo_da_curva = acha_unidades_discrepantes(df_faturamento)

if len(hoteis_acima_da_curva) > 0:
    print("\n--- Unidades com Faturamento Discrepante PARA CIMA ---")
    print(hoteis_acima_da_curva)
else:
    print("\n--- Não há unidades com faturamento discrepante para CIMA. ---")

if len(hoteis_abaixo_da_curva) > 0:
    print("\n--- Unidades com Faturamento Discrepante PARA BAIXO ---")
    print(hoteis_abaixo_da_curva)
else:
    print("\n--- Não há unidades com faturamento discrepante para BAIXO. ---")

#### 4. Relação entre RevPAR e avaliação dos hóspedes

* Cálculo do RevPAR por unidade (faturamento ÷ número de quartos).

* Correlação entre avaliação média e RevPAR. (Recurso: `.corr()`)

* Classificação da força da correlação segundo os intervalos de Pearson. (Recursos: função própria com `if`/`elif` - ver dicionário)

Obs: optamos pelo cálculo bruto da receita por quarto desconsiderando o padrão que utiliza as diárias. A métrica bruta permitiu identificar gargalos globais da rede.  

In [ ]:
avaliacao_unidade = reservas_realizadas.groupby('nome_unidade')['avaliacao_hospede'].mean()
quartos_unidade = reservas_realizadas.groupby('nome_unidade')['num_quartos_total'].first()
revpar_unidade = faturamento_unidade / (quartos_unidade)

df_estatistica = pd.DataFrame({
    'avaliacao': avaliacao_unidade,
    'revpar': revpar_unidade
}).dropna()

corr_valor = df_estatistica['avaliacao'].corr(df_estatistica['revpar'])

print(f"Total de unidades incluídas na análise: {len(df_estatistica)} de {len(avaliacao_unidade)}\n")

print("--------------------- Estatísticas das Avaliações dos Hóspedes ---")
print(f"Média Geral da Rede : {df_estatistica['avaliacao'].mean():.2f}")
print(f"Desvio Padrão       : {df_estatistica['avaliacao'].std():.2f}\n")

print("\n--- Matriz de Correlação ---")
print(df_estatistica[['avaliacao', 'revpar']].corr())

print(f"\n--- Diagnóstico Executivo ---")
print(f"Resultado: {classificar_correlacao(corr_valor)}")

Identificando alguns exemplos concretos da relavação RevPAR e Avaliação

In [ ]:
unidades_alvo = ['NaraHoteis Nova Iguaçu Centro', 
                 'NaraHoteis Copacabana Premium', 
                 'NaraHoteis Centro',
                 'NaraHoteis Nova Iguaçu Park']

avaliacoes_filtradas = avaliacao_unidade.loc[unidades_alvo]

print("---- Avaliação Média dos usuários por Unidade Específica ---")
display(avaliacoes_filtradas.round(2))


#### 5. Estimativa de RevPAR a partir da avaliação média

#### 6. Variabilidade regional do faturamento

* Faturamento total por região e por cidade.

* Coeficiente de variação por região (desvio padrão ÷ média), para comparar a instabilidade relativa do faturamento entre regiões de portes diferentes.

* Apesar do cáculo inicial do faturamento mensal, optamos por utilizar no cálculo abaixo, 
a `variável reservas realizadas` que restringe os valores somente às reservas `confirmadas` e `concluidas`

In [ ]:

faturamento_mensal_regiao = reservas_realizadas.groupby(['regiao', 'mes'])['faturamento'].sum().reset_index()
faturamento_mensal_unidade = reservas_realizadas.groupby(['nome_unidade', 'mes'])['faturamento'].sum().reset_index()

media_regiao = faturamento_mensal_regiao.groupby('regiao')['faturamento'].mean()
desvio_regiao = faturamento_mensal_regiao.groupby('regiao')['faturamento'].std()
coef_variacao_regiao = (desvio_regiao / media_regiao * 100).round(2)

media_unidade = faturamento_mensal_unidade.groupby('nome_unidade')['faturamento'].mean()
desvio_unidade = faturamento_mensal_unidade.groupby('nome_unidade')['faturamento'].std()
coef_variacao_unidades = (desvio_unidade / media_unidade * 100).round(2)

print("\n-------- Faturamento médio mensal por região  ---")
display(media_regiao.apply(converte_moeda))

print("\n-------- Faturamento médio mensal por unidades ---")
display(media_unidade.apply(converte_moeda))

print("--------- Coef. de V do Faturamento por Região ---")
display(coef_variacao_regiao.sort_values(ascending=False))

print("\n-------- Coef. de Variação do Faturamento por Unidade ---")
display(coef_variacao_unidades.sort_values(ascending=False))

### Projeção Gráfica

In [ ]:
faturamento_unidade.index = (
    faturamento_unidade.index
    .str.replace('NaraHoteis ', '', regex=False))

fig1, eixos1 = plt.subplots(1, 2, figsize=(14, 5))

eixos1[0].bar(faturamento_unidade.index, faturamento_unidade.values)
eixos1[0].set_title('Faturamento por Unidade')
eixos1[0].tick_params(axis='x', rotation=90)

eixos1[1].bar(faturamento_regiao.index, faturamento_regiao.values)
eixos1[1].set_title('Faturamento por Região')
eixos1[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
fig2, eixos2 = plt.subplots(1, 2, figsize=(14, 5))

eixos2[0].bar(
    total_excedente_por_unidade.index.str.replace('NaraHoteis ', '', regex=False), 
    total_excedente_por_unidade.values)
eixos2[0].set_title('Hóspedes Excedentes por Unidade')
eixos2[0].tick_params(axis='x', rotation=90)

avaliacao_ordenada = avaliacao_unidade.sort_values()

avaliacao_ordenada.index = (
    avaliacao_ordenada.index
    .str.replace('NaraHoteis ', '', regex=False))

eixos2[1].plot(avaliacao_ordenada.index, avaliacao_ordenada.values, marker='o')
eixos2[1].axhline(y=avaliacao_unidade.mean(), linestyle='--')
eixos2[1].set_title('Avaliação Média por Unidade')
eixos2[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

Gráfico de Faturamento Mensal Por Região e Unidade

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(faturamento_mensal_regiao['faturamento'], bins=25)

axes[0].axvline(
    faturamento_mensal_regiao['faturamento'].mean(),
    label=f"Média: {faturamento_mensal_regiao['faturamento'].mean():.1f}"
)

axes[0].axvline(
    faturamento_mensal_regiao['faturamento'].median(),
    linestyle='--',
    label=f"Mediana: {faturamento_mensal_regiao['faturamento'].median():.1f}"
)

axes[0].set_title(
    f"Faturamento por Região\n"
    f"Assimetria: {faturamento_mensal_regiao['faturamento'].skew():.2f} | "
    f"Curtose: {faturamento_mensal_regiao['faturamento'].kurt():.2f}"
)

axes[0].set_xlabel('Faturamento mensal')
axes[0].legend()



axes[1].hist(faturamento_mensal_unidade['faturamento'], bins=25)

axes[1].axvline(
    faturamento_mensal_unidade['faturamento'].mean(),
    label=f"Média: {faturamento_mensal_unidade['faturamento'].mean():.1f}"
)

axes[1].axvline(
    faturamento_mensal_unidade['faturamento'].median(),
    linestyle='--',
    label=f"Mediana: {faturamento_mensal_unidade['faturamento'].median():.1f}"
)

axes[1].set_title(
    f"Faturamento por Unidade\n"
    f"Assimetria: {faturamento_mensal_unidade['faturamento'].skew():.2f} | "
    f"Curtose: {faturamento_mensal_unidade['faturamento'].kurt():.2f}"
)

axes[1].set_xlabel('Faturamento mensal')
axes[1].legend()

plt.tight_layout()
plt.show()